# Data Profiling — Urban Heat Risk Mapping (Manhattan)

## Purpose

This notebook profiles the primary datasets used in the project:

- Landsat 8 Land Surface Temperature (LST)
- Sentinel-2 NDVI

The goal is to evaluate:

- spatial coverage
- temporal consistency
- cloud contamination
- statistical distributions
- dataset limitations
- suitability for urban heat risk analysis

The profiling process supports:

- `docs/problem-brief.md`
- `docs/problem-brief-v2.md`
- `docs/data-quality-audit.md`
- Session 3 preprocessing and modeling decisions

---

## Study Area

Manhattan, New York City, USA

---

## Time Period

Summer 2018  
(June 1, 2018 → August 31, 2018)

---

## Datasets Used

### 1. Landsat 8 Collection 2 Level 2

Used for:
- Land Surface Temperature (LST)
- Urban heat hotspot identification

### 2. Sentinel-2 Surface Reflectance

Used for:
- NDVI calculation
- Vegetation analysis
- Heat mitigation relationship analysis

---

## Expected Outputs

This notebook produces:

- LST statistics
- NDVI statistics
- heat-risk visualizations
- hotspot identification
- dataset quality observations
- profiling findings for Session 2 documentation

In [1]:
# ============================================
# IMPORTS
# ============================================

import ee
import geemap
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ============================================
# GOOGLE EARTH ENGINE AUTHENTICATION
# ============================================

ee.Authenticate()

# Initialize Earth Engine
ee.Initialize(project='ee-jineshnarendrajain')

print("Earth Engine initialized successfully")

Earth Engine initialized successfully


In [2]:
# ============================================
# DEFINE STUDY AREA
# ============================================

# Load NYC county boundaries
nyc = ee.FeatureCollection("TIGER/2018/Counties")

# Extract Manhattan geometry
manhattan = nyc.filter(
    ee.Filter.eq('NAME', 'New York')
).geometry()

print("Study area loaded successfully")

Study area loaded successfully


In [3]:
# ============================================
# LOAD LANDSAT 8 DATA
# ============================================

landsat = ee.ImageCollection(
    "LANDSAT/LC08/C02/T1_L2"
) \
.filterBounds(manhattan) \
.filterDate('2018-06-01', '2018-08-31') \
.filter(ee.Filter.lt('CLOUD_COVER', 20))

# Number of scenes
landsat_count = landsat.size().getInfo()

print(f"Landsat scenes loaded: {landsat_count}")

Landsat scenes loaded: 2


In [4]:
# ============================================
# COMPUTE LAND SURFACE TEMPERATURE (LST)
# ============================================

def get_lst(image):

    # Convert thermal band to Celsius
    lst = image.select('ST_B10') \
        .multiply(0.00341802) \
        .add(149.0) \
        .subtract(273.15) \
        .rename('LST')

    return lst.copyProperties(
        image,
        ['system:time_start']
    )

# Apply LST conversion
lst_collection = landsat.map(get_lst)

# Mean summer LST
lst_mean = lst_collection.mean().clip(manhattan)

print("LST layer generated successfully")

LST layer generated successfully


## Land Surface Temperature (LST)

Land Surface Temperature is derived from the thermal infrared band
(ST_B10) of Landsat 8 Collection 2 Level 2 imagery.

The thermal band values are converted into degrees Celsius using
the scaling factors provided by USGS.

LST is used in this project as a proxy for urban heat intensity.

### Why LST matters

Higher LST values typically correspond to:

- dense built-up urban areas
- impervious surfaces
- reduced vegetation
- urban heat island conditions

### Important limitation

LST measures surface temperature, NOT human thermal comfort.
It does not directly account for:

- humidity
- wind
- radiation balance
- shading

In [5]:
# ============================================
# LOAD SENTINEL-2 DATA
# ============================================

sentinel = ee.ImageCollection(
    "COPERNICUS/S2_SR"
) \
.filterBounds(manhattan) \
.filterDate('2018-06-01', '2018-08-31') \
.filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 10))

# Number of scenes
sentinel_count = sentinel.size().getInfo()

print(f"Sentinel-2 scenes loaded: {sentinel_count}")

c:\Users\User\anaconda3\Lib\site-packages\ee\deprecation.py:215: DeprecationWarning: 

Attention required for COPERNICUS/S2_SR! You are using a deprecated asset.
To make sure your code keeps working, please update it.
This dataset has been superseded by COPERNICUS/S2_SR_HARMONIZED

Learn more: https://developers.google.com/earth-engine/datasets/catalog/COPERNICUS_S2_SR

  warnings.warn(warning, category=DeprecationWarning)


Sentinel-2 scenes loaded: 7


## Sentinel-2 Vegetation Data

Sentinel-2 Surface Reflectance imagery is used to calculate
Normalized Difference Vegetation Index (NDVI).

The dataset provides:

- 10m spatial resolution
- multispectral optical imagery
- high-frequency revisit coverage

### Why NDVI matters

NDVI is used as a proxy for vegetation presence and density.

Higher NDVI values generally indicate:

- healthier vegetation
- denser green cover
- stronger cooling potential

Lower NDVI values generally correspond to:

- impervious urban surfaces
- sparse vegetation
- dense built environments

### Important limitation

NDVI does not directly measure:

- canopy shading
- vegetation height
- cooling intensity
- biodiversity

In [6]:
# ============================================
# CLOUD MASKING
# ============================================

def mask_s2_clouds(image):

    scl = image.select('SCL')

    # Remove:
    # 3 = cloud shadow
    # 8 = cloud medium probability
    # 9 = cloud high probability
    # 10 = thin cirrus
    # 11 = snow

    mask = scl.neq(3) \
        .And(scl.neq(8)) \
        .And(scl.neq(9)) \
        .And(scl.neq(10)) \
        .And(scl.neq(11))

    return image.updateMask(mask)

# Apply masking
sentinel_clean = sentinel.map(mask_s2_clouds)

print("Cloud masking completed")

Cloud masking completed


## Cloud Filtering & Data Quality

Cloud contamination is one of the major challenges in satellite-based
environmental analysis.

Clouds and cloud shadows can distort:

- NDVI values
- surface reflectance
- vegetation interpretation

To reduce these errors, the Sentinel-2 Scene Classification Layer (SCL)
is used to mask:

- cloud shadows
- medium/high probability clouds
- cirrus clouds
- snow pixels

### Remaining limitation

Even after masking:

- some cloud artifacts may remain
- aggressive masking may remove valid observations
- temporal gaps may still occur due to revisit frequency

In [7]:
# ============================================
# COMPUTE NDVI
# ============================================

def get_ndvi(image):

    ndvi = image.normalizedDifference(
        ['B8', 'B4']
    ).rename('NDVI')

    return ndvi.copyProperties(
        image,
        ['system:time_start']
    )

# Apply NDVI calculation
ndvi_collection = sentinel_clean.map(get_ndvi)

# Median summer NDVI
ndvi_mean = ndvi_collection.median().clip(manhattan)

print("NDVI layer generated successfully")

NDVI layer generated successfully


## NDVI Calculation

Normalized Difference Vegetation Index (NDVI) is calculated using:

NDVI = (NIR − RED) / (NIR + RED)

Where:

- NIR = Near Infrared band (B8)
- RED = Red spectral band (B4)

### Interpretation

| NDVI Value | Interpretation |
|---|---|
| < 0 | Water / impervious surfaces |
| 0 – 0.2 | Sparse vegetation |
| 0.2 – 0.5 | Moderate vegetation |
| > 0.5 | Dense healthy vegetation |

### Role in this project

NDVI is used to explain variations in urban surface temperature and to
identify areas with limited vegetation cover that may contribute to
higher heat exposure.

In [8]:
# ============================================
# LST STATISTICS
# ============================================

lst_stats = lst_mean.reduceRegion(
    reducer=ee.Reducer.minMax().combine(
        reducer2=ee.Reducer.mean(),
        sharedInputs=True
    ),
    geometry=manhattan,
    scale=30,
    maxPixels=1e9
)

lst_results = lst_stats.getInfo()

print("===== LST STATISTICS =====")
print(f"Minimum LST : {lst_results['LST_min']:.2f} °C")
print(f"Mean LST    : {lst_results['LST_mean']:.2f} °C")
print(f"Maximum LST : {lst_results['LST_max']:.2f} °C")

===== LST STATISTICS =====
Minimum LST : 10.33 °C
Mean LST    : 30.73 °C
Maximum LST : 49.20 °C


## Interpretation of LST Statistics

The extracted LST statistics provide an overview of thermal conditions
across Manhattan during Summer 2018.

### Observations

- High maximum temperatures indicate strong urban heat island effects
- Dense built-up areas are expected to contribute to elevated temperatures
- Spatial variability suggests uneven heat exposure across neighborhoods

### Important consideration

Extreme values may be influenced by:

- impervious materials
- rooftop heating
- transportation infrastructure
- residual cloud contamination

In [9]:
# ============================================
# NDVI STATISTICS
# ============================================

ndvi_stats = ndvi_mean.reduceRegion(
    reducer=ee.Reducer.minMax().combine(
        reducer2=ee.Reducer.mean(),
        sharedInputs=True
    ),
    geometry=manhattan,
    scale=10,
    maxPixels=1e9
)

ndvi_results = ndvi_stats.getInfo()

print("===== NDVI STATISTICS =====")
print(f"Minimum NDVI : {ndvi_results['NDVI_min']:.2f}")
print(f"Mean NDVI    : {ndvi_results['NDVI_mean']:.2f}")
print(f"Maximum NDVI : {ndvi_results['NDVI_max']:.2f}")

===== NDVI STATISTICS =====
Minimum NDVI : -0.27
Mean NDVI    : 0.09
Maximum NDVI : 0.70


## Interpretation of NDVI Statistics

The NDVI statistics summarize vegetation conditions across Manhattan
during the selected summer period.

### Observations

- Most urban areas exhibit relatively low NDVI values
- Higher NDVI values are concentrated in parks and waterfront vegetation
- Central Park is expected to dominate the highest NDVI ranges

### Important consideration

Low NDVI does not always indicate environmental failure.

Dense urban areas naturally produce:

- low vegetation coverage
- high impervious surface ratios
- reduced reflectance associated with vegetation

In [10]:
# ============================================
# VISUALIZATION MAP
# ============================================

Map = geemap.Map(
    center=[40.78, -73.97],
    zoom=12
)

# LST visualization parameters
lst_vis = {
    'min': 20,
    'max': 40,
    'palette': ['blue', 'green', 'yellow', 'red']
}

# NDVI visualization parameters
ndvi_vis = {
    'min': 0.1,
    'max': 0.8,
    'palette': ['brown', 'yellow', 'green']
}

# Add layers
Map.addLayer(lst_mean, lst_vis, 'LST')
Map.addLayer(ndvi_mean, ndvi_vis, 'NDVI')

# Display map
Map

Map(center=[40.78, -73.97], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright…

## Spatial Visualization

The map visualizes the spatial distribution of:

- Land Surface Temperature (LST)
- Vegetation (NDVI)

### Expected spatial patterns

#### High LST zones

Likely concentrated in:

- Midtown Manhattan
- Lower Manhattan
- dense transportation corridors
- highly impervious urban areas

#### High NDVI zones

Likely concentrated in:

- Central Park
- waterfront green spaces
- recreational open spaces

### Interpretation goal

The visualization supports identification of:

- urban heat hotspots
- vegetation deficits
- potential intervention priority zones

In [11]:
# ============================================
# HEAT RISK LAYER
# ============================================

# Normalize LST
lst_min = ee.Number(lst_results['LST_min'])
lst_max = ee.Number(lst_results['LST_max'])

lst_normalized = lst_mean.subtract(lst_min) \
    .divide(lst_max.subtract(lst_min))

# Normalize NDVI
ndvi_min = ee.Number(ndvi_results['NDVI_min'])
ndvi_max = ee.Number(ndvi_results['NDVI_max'])

ndvi_normalized = ndvi_mean.subtract(ndvi_min) \
    .divide(ndvi_max.subtract(ndvi_min))

# Risk logic:
# high LST increases risk
# high NDVI reduces risk

risk = lst_normalized.subtract(
    ndvi_normalized
).rename('Risk')

print("Heat risk layer generated")

Heat risk layer generated


## Heat Risk Classification Logic

A simplified rule-based risk model is used to estimate relative heat
exposure across Manhattan.

### Risk Formula

Risk = normalized LST − normalized NDVI

### Interpretation

| Condition | Risk Outcome |
|---|---|
| High LST + Low NDVI | High risk |
| Low LST + High NDVI | Low risk |

### Why normalization is required

LST and NDVI operate on different numerical ranges.

Normalization converts both datasets into comparable scales before
combining them.

### Important limitation

This is a simplified relative risk indicator and NOT a predictive
thermal comfort model.

The model does not include:

- humidity
- wind
- shading geometry
- demographic vulnerability

In [12]:
# ============================================
# RISK VISUALIZATION
# ============================================

risk_vis = {
    'min': -1,
    'max': 1,
    'palette': ['green', 'yellow', 'red']
}

Map.addLayer(risk, risk_vis, 'Heat Risk')

Map

Map(center=[40.78, -73.97], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright…

## Heat Risk Visualization

The heat risk layer combines:

- elevated surface temperature
- reduced vegetation presence

to identify relative urban heat exposure zones.

### Color interpretation

| Color | Meaning |
|---|---|
| Green | Lower relative heat risk |
| Yellow | Moderate heat risk |
| Red | Higher relative heat risk |

### Expected hotspot areas

Higher-risk zones are expected in:

- dense commercial districts
- transportation-heavy corridors
- highly impervious urban surfaces

Lower-risk zones are expected in:

- large vegetated areas
- parks
- waterfront green corridors

In [13]:
# ============================================
# HOTSPOT EXTRACTION
# ============================================

# Extract highest-risk pixels
hotspots = risk.sample(
    region=manhattan,
    scale=30,
    numPixels=10,
    geometries=True
)

# Convert to GeoJSON-style dictionary
hotspot_info = hotspots.getInfo()

print("Top hotspot samples extracted")
print(f"Number of hotspots: {len(hotspot_info['features'])}")

Top hotspot samples extracted
Number of hotspots: 10


## Hotspot Identification

The system extracts high-risk sample locations from the generated risk
surface.

These hotspots represent areas with:

- elevated land surface temperature
- limited vegetation presence

### Intended use

The extracted zones support:

- heat mitigation prioritization
- early-stage planning decisions
- identification of intervention targets

### Example interventions

Potential mitigation strategies include:

- tree planting
- shading infrastructure
- reflective materials
- increased green cover

### Important limitation

The hotspots are derived from raster-based approximations and should
not be interpreted as precise street-level thermal measurements.

# Final Profiling Findings

## Key Findings

### 1. Strong Urban Heat Island Patterns

Manhattan exhibits clear spatial concentration of elevated land surface
temperature in dense built-up regions.

Commercial and transportation-heavy areas show consistently higher
surface temperatures.

---

### 2. Vegetation Concentration Is Uneven

NDVI values indicate that vegetation is concentrated primarily in:

- Central Park
- waterfront zones
- isolated green spaces

Most dense urban areas exhibit relatively low vegetation coverage.

---

### 3. Inverse Relationship Between NDVI and LST

Areas with higher vegetation generally correspond to lower land surface
temperature values.

This supports the role of vegetation in mitigating urban heat exposure.

---

### 4. Satellite Data Is Suitable for Relative Heat Analysis

The datasets are suitable for:

- identifying relative heat exposure patterns
- comparing vegetation influence
- supporting neighborhood-scale prioritization

However, they remain proxy-based representations rather than direct
measurements of human thermal comfort.

---

# Implications for Session 3

The Session 3 workflow must include:

- reproducible cloud filtering
- temporal consistency checks
- raster alignment
- normalized risk scoring
- transparent rule-based classification
- validation against NYC Heat Vulnerability Index

---

# Major Limitations

The analysis does NOT account for:

- humidity
- wind
- urban canyon effects
- shading geometry
- indoor thermal conditions
- demographic vulnerability

Results should therefore be interpreted as relative environmental heat
risk rather than complete thermal comfort assessment.

In [14]:
# ============================================
# SAVE PROFILE SUMMARY
# ============================================

profile_summary = {
    "study_area": "Manhattan",
    "time_period": "2018-06-01 to 2018-08-31",
    "landsat_scene_count": landsat_count,
    "sentinel_scene_count": sentinel_count,
    "lst_statistics": lst_results,
    "ndvi_statistics": ndvi_results
}

import json

with open("../data/profile-summary.json", "w") as f:
    json.dump(profile_summary, f, indent=2)

print("profile-summary.json saved successfully")

profile-summary.json saved successfully
